# Scenario A — Elasticsearch → DB (Bulk Create)

Records found **in Elasticsearch but missing from PostgreSQL**.  
Run each step in order. Review and download data at each stage before moving on.

| Step | Action |
|------|--------|
| 1 | Load session from `1_compare.ipynb` + enter credentials |
| 2 | View ES-not-in-DB records |
| 3 | Filter already-deleted records |
| 4 | Find & remove test data (marks deleted in DB via SQL) |
| 5 | Verify beneficiaries exist in system |
| 6 | Build API payload (fetch from ES + transform) |
| 7 | Run Bulk Create |
| — | Retry failed records (if needed) |

In [ ]:
import base64, json, os, sys, warnings
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML
warnings.filterwarnings('ignore')

repo_root = Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline import compare as _cmp
from pipeline import filter as _flt
from pipeline import verify as _ver
from pipeline import transform as _trn
from pipeline import ingest as _ing

with open('campaign_config.json') as f:
    _raw = json.load(f)
_GLOBALS  = _raw.get('_globals', {})
CAMPAIGNS = {k: v for k, v in _raw.items() if k != '_globals'}

def get_cfg(key):
    return {**_GLOBALS, **CAMPAIGNS[key]}

def _dl(path, label='Download'):
    if path and os.path.exists(str(path)):
        display(FileLink(str(path), result_html_prefix=f'\u2b07  {label}: '))

# Shared state across steps
SESSION      = {}
CFG          = {}
DB_CONFIG    = {}
ES_RAW_DF    = pd.DataFrame()   # loaded from compare output
ES_ACTIVE_DF = pd.DataFrame()   # after isDeleted filter
ES_CLEAN_DF  = pd.DataFrame()   # after test data removal
VERIFIED_DF  = pd.DataFrame()   # after beneficiary check
TASKS_PAYLOAD = []

print('Setup complete. Run the cells below in order.')

In [ ]:
_W = {'description_width': '110px'}

w_campaign = widgets.Dropdown(
    options=[(v['label'], k) for k, v in CAMPAIGNS.items()],
    description='Campaign:', style=_W, layout=widgets.Layout(width='340px')
)
w_outdir = widgets.Text(value='output/', description='Output dir:', style=_W, layout=widgets.Layout(width='340px'))

btn_load = widgets.Button(description='Load', button_style='info', icon='folder-open',
                          layout=widgets.Layout(width='160px'))
out_load = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _load(b):
    global SESSION, CFG, DB_CONFIG
    with out_load:
        out_load.clear_output()
        try:
            key = w_campaign.value
            CFG = get_cfg(key)
            DB_CONFIG = {
                'host': CFG['db_host'], 'port': CFG['db_port'], 'database': CFG['db_name'],
                'user': CFG['db_user'], 'password': CFG['db_pass'],
                'sslmode': 'require', 'connect_timeout': 30,
            }

            base = w_outdir.value.strip().rstrip('/')
            output_dir = os.path.join(base, key) if os.path.isdir(os.path.join(base, key)) else base

            if not os.path.isdir(output_dir):
                print(f'ERROR: Directory not found: {output_dir}')
                print('Run 1_compare.ipynb first.')
                return

            # Check compare was run — look for any Elastic_not_in_DB file (always created)
            marker_files = sorted(f for f in os.listdir(output_dir) if f.startswith('Elastic_not_in_DB_'))
            if not marker_files:
                print(f'ERROR: No compare output found in {output_dir}')
                print('Run 1_compare.ipynb first.')
                return

            from datetime import datetime
            SESSION = {
                'campaign_key': key,
                'output_dir':   output_dir,
                'ts':           datetime.now().strftime('%Y%m%d_%H%M%S'),
            }

            display(HTML(
                f"<div style='font-family:monospace;background:#f3fff3;padding:10px;border-radius:4px;border:1px solid #bdb'>"
                f"✓ Ready<br>"
                f"Campaign  : <b>{key}</b> ({CFG['label']})<br>"
                f"Output dir: {output_dir}<br>"
                f"Latest run: {marker_files[-1]}<br>"
                f"DB: {CFG['db_host']} / {CFG['db_name']} as {CFG['db_user']}"
                f"</div>"
            ))
        except Exception:
            import traceback; traceback.print_exc()

btn_load.on_click(_load)
display(widgets.HTML('<h3>Setup</h3>'))
display(w_campaign, w_outdir, btn_load, out_load)

In [ ]:
btn_s1 = widgets.Button(description='Step 1 — Load ES Data', button_style='primary', icon='eye',
                        layout=widgets.Layout(width='260px'))
out_s1 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step1(b):
    global ES_RAW_DF
    with out_s1:
        out_s1.clear_output()
        try:
            output_dir = SESSION['output_dir']
            all_files  = os.listdir(output_dir)

            # Prefer the details file; fall back to the ID-list file
            detail_files = sorted(f for f in all_files if f.startswith('Elastic_details_not_in_db'))
            id_files     = sorted(f for f in all_files if f.startswith('Elastic_not_in_DB_'))

            if detail_files:
                path = os.path.join(output_dir, detail_files[-1])
            elif id_files:
                path = os.path.join(output_dir, id_files[-1])
            else:
                print('ERROR: No Elastic compare output file found. Run 1_compare.ipynb first.')
                return

            ES_RAW_DF = pd.read_csv(path)
            print(f'Loaded {len(ES_RAW_DF):,} ES records not found in DB  ({os.path.basename(path)})')

            if len(ES_RAW_DF) == 0:
                print('No records missing from DB — nothing to do for Scenario A.')
                return

            print(f'Columns: {list(ES_RAW_DF.columns)}')
            display(ES_RAW_DF.head(20))
            _dl(path, 'Elastic-not-in-DB details')
        except Exception:
            import traceback; traceback.print_exc()

btn_s1.on_click(_step1)
display(widgets.HTML('<h3>Step 1 — View ES Records Not in DB</h3>'))
display(btn_s1, out_s1)

In [ ]:
btn_s2 = widgets.Button(description='Step 2 — Filter isDeleted', button_style='primary', icon='filter',
                        layout=widgets.Layout(width='260px'))
out_s2 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step2(b):
    global ES_ACTIVE_DF
    with out_s2:
        out_s2.clear_output()
        try:
            col = next((c for c in ES_RAW_DF.columns
                        if c.lower() in ('data.isdeleted', 'isdeleted')), None)
            if col:
                already_del = ES_RAW_DF[ES_RAW_DF[col] == True]
                ES_ACTIVE_DF = ES_RAW_DF[ES_RAW_DF[col] != True].copy()
                print(f'Already isDeleted  : {len(already_del):,}')
                print(f'Active (to process): {len(ES_ACTIVE_DF):,}')
                if len(already_del):
                    p = os.path.join(SESSION['output_dir'], f"removed_isDeleted_{SESSION['ts']}.xlsx")
                    already_del.to_excel(p, index=False)
                    _dl(p, 'Already-deleted records')
            else:
                print('No isDeleted column found — treating all records as active')
                ES_ACTIVE_DF = ES_RAW_DF.copy()
            display(ES_ACTIVE_DF.head(20))
        except Exception:
            import traceback; traceback.print_exc()

btn_s2.on_click(_step2)
display(widgets.HTML('<h3>Step 2 — Filter Already-Deleted Records</h3>'))
display(btn_s2, out_s2)

In [ ]:
btn_s3a = widgets.Button(description='3a — Find Test Data', button_style='warning', icon='search',
                         layout=widgets.Layout(width='220px'))
btn_s3b = widgets.Button(description='3b — Mark Deleted in DB (SQL)', button_style='danger', icon='trash',
                         layout=widgets.Layout(width='270px'))
btn_s3b.disabled = True
out_s3 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))
TEST_IDS = []

def _step3a(b):
    global ES_CLEAN_DF, TEST_IDS
    with out_s3:
        out_s3.clear_output()
        try:
            tuc      = CFG.get('test_user_config', {})
            prefixes = tuc.get('es_test_createdby_prefixes', [])
            ES_CLEAN_DF, test_df = _flt.filter_test_data_es(
                ES_ACTIVE_DF, prefixes, createdby_col='Data.createdBy'
            )
            print(f'Test records found  : {len(test_df):,}')
            print(f'Clean records remain: {len(ES_CLEAN_DF):,}')
            if len(test_df):
                # Determine the clientReferenceId column
                id_col = next((c for c in test_df.columns
                               if 'taskclientreferenceid' in c.lower()
                               or c.lower() == 'data.taskclientreferenceid'), None)
                TEST_IDS = test_df[id_col].dropna().tolist() if id_col else []
                display(HTML('<b>Test data preview (first 20 rows):</b>'))
                display(test_df.head(20))
                p = os.path.join(SESSION['output_dir'], f"test_data_es_{SESSION['ts']}.xlsx")
                test_df.to_excel(p, index=False)
                _dl(p, 'Test data')
                btn_s3b.disabled = False
                display(HTML("<br><span style='color:orange'>\u26a0 Review above. Click <b>3b</b> to mark these as deleted in the DB via SQL UPDATE.</span>"))
            else:
                print('No test data found. Proceed to Step 4.')
        except Exception:
            import traceback; traceback.print_exc()

def _step3b(b):
    btn_s3b.disabled = True
    with out_s3:
        try:
            if not TEST_IDS:
                print('No test IDs to mark.')
                return
            task_cfg = CFG['task']
            print(f'Running SQL UPDATE on {task_cfg["db_table"]} for {len(TEST_IDS):,} records...')
            updated = _flt.mark_deleted_in_db(
                DB_CONFIG, task_cfg['db_table'], task_cfg['db_id_column'], TEST_IDS
            )
            display(HTML(f"<span style='color:green'>\u2713 {updated:,} rows marked isdeleted=true in DB</span>"))
        except Exception:
            import traceback; traceback.print_exc()
            btn_s3b.disabled = False

btn_s3a.on_click(_step3a)
btn_s3b.on_click(_step3b)
display(widgets.HTML('<h3>Step 3 — Identify &amp; Remove Test Data</h3>'))
display(widgets.HTML("<small>Test records are identified by <code>Data.createdBy</code> prefix in ES (e.g. 'test', 'demo').</small>"))
display(widgets.HBox([btn_s3a, btn_s3b]), out_s3)

In [ ]:
btn_s4 = widgets.Button(description='Step 4 — Verify Beneficiaries', button_style='primary', icon='check',
                        layout=widgets.Layout(width='270px'))
out_s4 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step4(b):
    global VERIFIED_DF
    btn_s4.disabled = True
    with out_s4:
        out_s4.clear_output()
        try:
            ben_col = next(
                (c for c in ES_CLEAN_DF.columns
                 if 'projectbeneficiaryclientreferenceid' in c.lower()),
                None
            )
            if not ben_col:
                print('ERROR: Cannot find projectBeneficiaryClientReferenceId column in data')
                return

            all_ids = ES_CLEAN_DF[ben_col].dropna().unique().tolist()
            print(f'Verifying {len(all_ids):,} beneficiary IDs via API...')

            not_found = _ver.verify_beneficiaries(
                api_base   = CFG['api_base'],
                tenant_id  = CFG['tenant_id'],
                auth_token = CFG['auth_token'],
                ids        = all_ids,
            )

            VERIFIED_DF, removed = _ver.segregate(ES_CLEAN_DF, not_found, ben_col)

            display(HTML(
                f"<table style='border-collapse:collapse;font-family:monospace'>"
                f"<tr><td style='padding:4px 20px 4px 0'><b>Beneficiaries checked</b></td><td>{len(all_ids):,}</td></tr>"
                f"<tr><td>Present in system</td><td style='color:green'>{len(all_ids)-len(not_found):,}</td></tr>"
                f"<tr><td>Missing (excluded)</td><td style='color:{'red' if not_found else 'green'}'>{len(not_found):,}</td></tr>"
                f"<tr><td><b>Records to ingest</b></td><td><b>{len(VERIFIED_DF):,}</b></td></tr>"
                f"</table>"
            ))

            if not_found:
                mp = os.path.join(SESSION['output_dir'], f"missing_beneficiaries_{SESSION['ts']}.xlsx")
                pd.DataFrame({'missing_beneficiary_id': not_found}).to_excel(mp, index=False)
                _dl(mp, 'Missing beneficiaries')

            vp = os.path.join(SESSION['output_dir'], f"verified_to_ingest_{SESSION['ts']}.xlsx")
            VERIFIED_DF.to_excel(vp, index=False)
            _dl(vp, 'Verified records (to ingest)')
            display(VERIFIED_DF.head(20))
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn_s4.disabled = False

btn_s4.on_click(_step4)
display(widgets.HTML('<h3>Step 4 — Verify Beneficiaries Exist in System</h3>'))
display(btn_s4, out_s4)

In [ ]:
btn_s5 = widgets.Button(description='Step 5 — Build Payload', button_style='primary', icon='cogs',
                        layout=widgets.Layout(width='250px'))
out_s5 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step5(b):
    global TASKS_PAYLOAD
    btn_s5.disabled = True
    with out_s5:
        out_s5.clear_output()
        try:
            task_cfg = CFG['task']

            # Find taskClientReferenceId column in VERIFIED_DF
            id_col = next(
                (c for c in VERIFIED_DF.columns if 'taskclientreferenceid' in c.lower()),
                None
            )
            if not id_col:
                print('ERROR: Cannot find taskClientReferenceId column in verified data')
                return
            ids = VERIFIED_DF[id_col].dropna().tolist()
            print(f'Building payload for {len(ids):,} tasks...')

            auth_header = 'Basic ' + base64.b64encode(
                f"{CFG['es_username']}:{CFG['es_password']}".encode()
            ).decode()

            all_tasks = _trn.fetch_and_transform(
                es_base       = CFG['es_base_url'],
                es_index      = task_cfg['es_index'],
                es_auth_header= auth_header,
                task_ids      = ids,
            )

            valid, issues = _trn.validate_payload(all_tasks)
            TASKS_PAYLOAD = valid

            print(f'\nValidation:')
            print(f'  Valid  : {len(valid):,}')
            print(f'  Issues : {len(issues):,}')

            if issues:
                ip = os.path.join(SESSION['output_dir'], f"payload_issues_{SESSION['ts']}.json")
                with open(ip, 'w') as f:
                    json.dump(issues, f, indent=2)
                _dl(ip, 'Payload issues')

            pp = os.path.join(SESSION['output_dir'], f"payload_create_{SESSION['ts']}.json")
            with open(pp, 'w') as f:
                json.dump(TASKS_PAYLOAD, f, indent=2)
            _dl(pp, 'Full payload JSON')

            display(HTML('<br><b>Payload preview (first 2 records):</b>'))
            display(HTML(f"<pre style='background:#f9f9f9;padding:8px;max-height:300px;overflow:auto'>{json.dumps(TASKS_PAYLOAD[:2], indent=2)}</pre>"))
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn_s5.disabled = False

btn_s5.on_click(_step5)
display(widgets.HTML('<h3>Step 5 — Fetch from ES &amp; Build Create Payload</h3>'))
display(btn_s5, out_s5)

In [ ]:
btn_s6 = widgets.Button(description='Step 6 — Run Bulk Create', button_style='danger', icon='upload',
                        layout=widgets.Layout(width='260px'))
out_s6 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step6(b):
    btn_s6.disabled = True
    with out_s6:
        out_s6.clear_output()
        try:
            if not TASKS_PAYLOAD:
                print('ERROR: No payload ready. Run Step 5 first.')
                return
            api_url     = CFG['api_base'] + CFG['api_paths']['task_create']
            failed_path = os.path.join(SESSION['output_dir'], f"failed_create_{SESSION['ts']}.json")

            print(f'Sending {len(TASKS_PAYLOAD):,} tasks to:')
            print(f'  {api_url}\n')

            result = _ing.bulk_ingest(
                api_url     = api_url,
                auth_token  = CFG['auth_token'],
                tasks       = TASKS_PAYLOAD,
                failed_path = failed_path,
            )

            display(HTML(
                f"<div style='font-family:monospace;margin-top:8px'>"
                f"<span style='color:green'>\u2713 Success: {result['success']:,}</span><br>"
                f"<span style='color:{'red' if result['failed'] else 'green'}'>\u2715 Failed : {result['failed']:,}</span>"
                f"</div>"
            ))
            if result['failed']:
                _dl(failed_path, 'Failed records — use Retry cell below')
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn_s6.disabled = False

btn_s6.on_click(_step6)
display(widgets.HTML('<h3>Step 6 — Bulk Create (ES &#8594; DB)</h3>'))
display(widgets.HTML(
    "<div style='background:#fff3f3;padding:8px 12px;border-radius:4px;border:1px solid #fcc;margin-bottom:6px'>"
    "\u26a0 This writes data to the HCM API. Confirm Steps 1&#8211;5 are correct before clicking."
    "</div>"
))
display(btn_s6, out_s6)

In [ ]:
btn_retry = widgets.Button(description='Retry Failed Records', button_style='warning', icon='refresh',
                           layout=widgets.Layout(width='230px'))
out_retry = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _retry(b):
    btn_retry.disabled = True
    with out_retry:
        out_retry.clear_output()
        try:
            failed_path = os.path.join(SESSION['output_dir'], f"failed_create_{SESSION['ts']}.json")
            if not os.path.exists(failed_path):
                print('No failed_create file found.')
                return
            api_url = CFG['api_base'] + CFG['api_paths']['task_create']
            result  = _ing.retry_failed(failed_path, api_url, CFG['auth_token'])
            print(f"Retry — Success: {result['success']:,} | Failed: {result['failed']:,}")
            if result['failed']:
                _dl(failed_path, 'Still-failed records')
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn_retry.disabled = False

btn_retry.on_click(_retry)
display(widgets.HTML('<h3>Retry Failed Records (optional)</h3>'))
display(btn_retry, out_retry)